# Beyond First Attendance: Institutional Incentives, High-Frequency Mathematics Support Use, and Heterogeneous Academic Outcomes

**Paper 2.1 — executable notebook version**

This notebook mirrors the scientific argument of the Paper 2.1 manuscript while exposing the code that produces the reported quantities. It is intended to be read from top to bottom as a computational paper: institutional context and hypotheses are presented in Markdown; the corresponding cohort construction, estimators, tables, and diagnostics appear immediately afterward in executable Python cells.

The notebook intentionally reuses the same `cmat_analysis` API and the same Paper 2.1 grouping rules as the manuscript. It does **not** maintain a separate statistical implementation.

> **Interpretive rule.** CMAT attendance is observational. The notebook describes outcome differences associated with recorded attendance and visit frequency; it does not identify a causal dose-response effect of additional visits.


## Abstract

Mathematics-support attendance is often summarized as a binary indicator, although repeated use may reflect different mixtures of academic need, voluntary help-seeking, and institutional encouragement. We analyze 6,627 first eligible attempts at a first-year university mathematics course, linking support-centre visits to the same academic period and standardizing final performance within instructor-period groups.

The main reporting grid distinguishes 0, exact 1–6, and 7+ visits. During at least part of the study period, three CMAT visits could satisfy a first-semester PPA participation activity, so the first three visits cannot be interpreted as a purely voluntary dose of support. The paper therefore separates the large **zero-versus-any-attendance margin** from the structure of repeated use among students who attend.

The current results show a strong adjusted 0-versus-1+ association in both standardized performance and pass probability, but no monotone visit-by-visit staircase among users. The 7+ group has the highest observed pass rate and is the clearest high-frequency profile. Distributional analyses further show that administrative outcomes, especially BV, materially shape the lower tail of the completed continuous outcome. These patterns remain observational and should not be interpreted as causal effects of CMAT attendance.


## Introduction and literature framing

Mathematics and statistics support has become a routine component of higher education, particularly where students enter quantitatively demanding programmes with heterogeneous mathematical preparation. The literature nevertheless makes evaluation difficult because attendance is behavioral rather than assigned: students may attend because they are struggling, because they are motivated to improve, because they already value academic support, or because institutional structures encourage participation.

Much of the observational mathematics-support literature therefore concentrates on a first margin—**users versus non-users**—and frequently reports more favorable outcomes among students who attend. At the same time, identification-oriented work using randomized encouragement, instrumental variables, or stronger quasi-experimental adjustment has produced more mixed evidence about direct academic effects. Paper 2.1 treats this distinction as central: a large user/non-user association can be institutionally important without showing what would happen if an otherwise identical student were assigned an additional visit.

Repeated use introduces a second margin. Prior work, including the longitudinal mathematics-support literature cited in the manuscript, has separated one-time engagement from higher-frequency attendance. In the present setting this distinction is especially important because the first three CMAT visits may have been partly influenced by PPA participation credit. Visit frequency is therefore not modeled as a homogeneous treatment dose.

A third issue is distributional. Average grades can conceal a lower tail containing numeric failures and administrative course endings. For this reason, the paper studies both observed numeric grades and the completed continuous outcome after the pre-specified numerical treatment of BV, RT, and BA, and it compares two-Gaussian mixtures with a flexible one-component skew-normal alternative rather than equating non-normality with latent classes.

The notebook follows the manuscript's evidentiary hierarchy: broad engagement margin first, frequency among users second, observed adverse-outcome composition third, and distribution-shape diagnostics last. The full bibliography remains in \`paper/references.bib\`, which is displayed at the end.


## 1. Reproducibility contract

The canonical analysis has three layers:

1. `code/run_paper21.py` — outcome-blind support audit used to choose a defensible upper-tail grouping.
2. `code/run_paper21_outcomes.py` — benchmark, frequency, PASS, outcome-composition, and robustness analyses.
3. `code/run_paper21_mixture.py` — numeric-failure, Gaussian-mixture, skew-normal, bootstrap, and cross-validation analyses.

The notebook reproduces the central light-to-moderate computations directly with the same reusable functions. Expensive mixture bootstrap/CV calculations are optional because the verified aggregate outputs are already versioned under `results/paper21/tables/`.


### 1.1 Preparación del entorno reproducible

Esta celda establece el entorno de ejecución de la notebook. Primero localiza automáticamente la raíz del repositorio buscando \`paper/visit_grouping_spec.json\`, de manera que las rutas no dependan de que Jupyter se abra desde una carpeta específica; después define las rutas canónicas de tablas y figuras, añade \`cmat_analysis\` al \`sys.path\` y configura la visualización de \`pandas\`.

No se realiza todavía ningún análisis estadístico. El objetivo es que todas las celdas posteriores lean exactamente los mismos archivos y módulos que los scripts del paper, evitando rutas locales o implementaciones paralelas.


In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import sys
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "paper" / "visit_grouping_spec.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the CMAT-research root. Run this notebook from inside the repository."
    )

REPO_ROOT = find_repo_root(Path.cwd())
PAPER_DIR = REPO_ROOT / "paper"
TABLES_DIR = REPO_ROOT / "results" / "paper21" / "tables"
FIGURES_DIR = REPO_ROOT / "results" / "paper21" / "figures"

# Make the shared editable library importable even when it has not been pip-installed.
sys.path.insert(0, str(REPO_ROOT / "cmat_analysis"))

print("Repository:", REPO_ROOT)
print("Paper 2.1 tables:", TABLES_DIR)
print("Paper 2.1 figures:", FIGURES_DIR)


### 1.2 Importación de la API estadística canónica

Aquí se importan las funciones reutilizables de \`cmat_analysis\` que construyen la cohorte, generan los outcomes y ejecutan los modelos. La notebook no vuelve a programar estimadores estadísticos desde cero; utiliza la misma API pública que consumen \`run_paper21_outcomes.py\` y \`run_paper21_mixture.py\`.

También se fijan los órdenes de las categorías \`0,1,2,3,4,5,6,7+\` y de los estados académicos, porque el orden importa para construir contrastes, matrices pairwise y figuras. Finalmente se importa \`figures_paper21.py\`, que contiene las recetas gráficas exactas del manuscrito.


In [ ]:
from cmat_analysis.cohorts import build_study_cohorts, load_and_clean_inputs
from cmat_analysis.config.study_config import get_study_config
from cmat_analysis.measures import add_academic_outcome_states, add_primary_outcomes
from cmat_analysis.statistics import (
    add_topcoded_visit_group,
    compare_univariate_shape_models,
    cross_validated_skew_normal_vs_gmm,
    fixed_effect_group_comparisons,
    fixed_effect_logistic_adjusted_probabilities,
    fixed_effect_logistic_group_comparisons,
    gaussian_mixture_component_summary,
    gaussian_mixture_model_selection,
    gaussian_mixture_responsibilities,
    group_outcome_summary,
    parametric_bootstrap_skew_normal_vs_gmm,
    soft_component_composition,
)

GROUP_ORDER = ["0", "1", "2", "3", "4", "5", "6", "7+"]
USER_ORDER = ["1", "2", "3", "4", "5", "6", "7+"]
STATE_ORDER = ["pass", "numeric_nonpass", "BV", "RT", "BA"]

group_spec = json.loads((PAPER_DIR / "visit_grouping_spec.json").read_text(encoding="utf-8"))
group_spec

# Import the canonical publication-figure module so the notebook renders
# the same Matplotlib recipes as the manuscript instead of embedding PDFs.
CODE_DIR = REPO_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import figures_paper21 as paper21_figures
paper21_figures.set_style()


### Inline figure generation

Every manuscript figure below is generated by Matplotlib during the notebook run. The notebook imports the canonical functions from \`code/figures_paper21.py\`, but temporarily replaces their file-saving hook so that the live \`Figure\` object is rendered inline instead of loading a previously generated PDF.

This means the visual recipe is shared with the paper while the notebook itself performs the plotting.


### 1.3 Renderizado inline sin duplicar la lógica de las figuras

Las funciones de \`figures_paper21.py\` normalmente terminan guardando un PDF y cerrando el objeto de Matplotlib. Esta función intercepta únicamente ese último paso: sustituye temporalmente \`_save()\` por una función que devuelve el objeto \`Figure\`, lo muestra dentro de Jupyter y después restaura el comportamiento original.

Por tanto, la notebook **sí genera la figura en esa ejecución**; no abre una imagen previamente guardada. Al mismo tiempo, ejes, colores, transformaciones, anotaciones y datos siguen viniendo de la receta canónica del paper.


In [ ]:
def render_paper_figure(plot_function):
    """Render one canonical Paper 2.1 Matplotlib figure inline.

    The publication plotting functions normally save a PDF and close the figure.
    Here we temporarily replace only that final output step; all data loading,
    transformations, annotations, scales, and plotting logic remain canonical.
    """
    original_save = paper21_figures._save

    def _capture(fig, name):
        fig.tight_layout()
        return fig

    paper21_figures._save = _capture
    try:
        fig = plot_function()
        display(fig)
        plt.close(fig)
        return None
    finally:
        paper21_figures._save = original_save


## 2. Institutional context and scientific question

CMAT is the mathematics support centre at Universidad de las Américas Puebla. Students register attendance when entering the centre, and each registration is counted as one visit. The records identify that a visit occurred, but they do not measure its duration, exact content, or the tutor involved.

The key institutional complication is PPA. During at least part of the study period, three CMAT visits could satisfy a first-semester PPA participation activity. Consequently, one, two, or three visits may combine academic demand with an institutional participation incentive. A fourth or later visit necessarily exceeds that specific three-visit option, although later attendance remains self-selected.

This motivates three interpretive regions:

$$
0 \quad | \quad 1,2,3 \quad | \quad 4+
$$

where the first region is no recorded CMAT use, the middle region is PPA-compatible low-frequency use, and the last region represents repeated attendance beyond that specific participation threshold. The manuscript also resolves a \`7+\` high-frequency tail when describing the full attendance profile.

The four substantive questions are:

1. Where do adjusted performance and PASS outcomes differ across attendance frequencies?
2. Is there evidence of a break exactly at the third visit?
3. Does the distribution of outcomes contain more than one performance regime?
4. Does the composition of adverse outcomes—numeric failure, BV, RT, and BA—differ across attendance patterns?


## 3. Data, cohort, and outcomes

The unit of analysis is the student's **first eligible observed attempt at Matemáticas Universitarias (MU)** in an academic period with CMAT attendance coverage. Visits are linked by student and academic period, regardless of the subject label recorded for the CMAT visit.

Two principal performance outcomes are used:

- **PASS**: numeric final grade \(\ge 7.5\); numeric grade below 7.5, BV, RT, and BA are non-PASS.
- **Continuous standardized performance**: the completed final-grade value is standardized within instructor-period groups,

$$
Z_{ic}=\frac{Y_{ic}-\bar Y_c}{s_c},
$$

where \(c\) denotes the instructor × academic-period grading context.

For the continuous primary outcome, administrative endings are numerically represented using the pre-specified Paper 2 imputation procedure. A numeric complete-case standardized outcome is retained as a sensitivity analysis so that the role of administrative-outcome imputation can be inspected explicitly.


### 3.1 Construcción de la cohorte analítica

Esta celda parte de los dos archivos controlados y pseudonimizados utilizados por el paper: \`Materias_pseudonymized.csv\` y \`Asesorias_pseudonymized.csv\`. \`load_and_clean_inputs()\` normaliza las fuentes y \`build_study_cohorts()\` aplica las reglas de inclusión hasta obtener \`mu_primary\`, es decir, la primera tentativa elegible de MU por estudiante dentro de periodos con cobertura de CMAT.

Después, \`add_primary_outcomes()\` construye PASS, la calificación estandarizada complete-case y la versión continua primaria con el tratamiento preespecificado de BA/BV/RT. \`add_academic_outcome_states()\` añade las categorías administrativas utilizadas más adelante.

Finalmente se crea \`P21_GROUP_7P\`, que codifica la secuencia de reporte del manuscrito:

\[
0,\;1,\;2,\;3,\;4,\;5,\;6,\;7+.
\]

Esta celda define la población sobre la que operan todos los modelos siguientes.


In [ ]:
# Load the exact controlled pseudonymized inputs used by the canonical Paper 2.1 runners.
base_config = get_study_config(REPO_ROOT)
config = replace(
    base_config,
    materias_path=(REPO_ROOT / "data" / "controlled" / "Materias_pseudonymized.csv").resolve(),
    asesorias_path=(REPO_ROOT / "data" / "controlled" / "Asesorias_pseudonymized.csv").resolve(),
)

data = load_and_clean_inputs(config)

mu = build_study_cohorts(data, config)["mu_primary"].copy()
mu = add_primary_outcomes(mu, config)
mu["PASS"] = mu["PASS"].astype(float)
mu = add_academic_outcome_states(mu)

# Main manuscript reporting grid: 0 / exact 1-6 / 7+.
mu = add_topcoded_visit_group(
    mu,
    visits_col="VISITS_CMAT_PERIOD",
    top_exact=6,
    include_zero=True,
    output_col="P21_GROUP_7P",
)

print(f"Study cohort N = {len(mu):,}")
print(f"CMAT users N = {int(mu['VISITS_CMAT_PERIOD'].gt(0).sum()):,}")
print(f"Instructor-period groups = {mu['CLASSROOM_ID'].nunique():,}")


## 4. Why the upper tail is grouped

The historical outcome-blind support audit selected `1 / 2 / 3 / 4 / 5 / 6+` as the most conservative positive-attendance inferential grouping. The current manuscript additionally displays exact 6 and 7+ because the scientific question concerns high-frequency use, but this extra resolution must not be rewritten as if it had been pre-specified from outcome significance.

The governing principle is support-preserving resolution: retain exact visit counts while student counts and overlap across instructor-period grading contexts remain adequate; pool once exact-frequency cells become too sparse for stable adjusted comparisons.


### 4.1 Auditoría de soporte antes de mirar outcomes

Aquí no se utilizan calificaciones ni PASS. Se leen dos tablas generadas por la auditoría **outcome-blind** que cuantifica, para cada frecuencia de visitas, cuántos estudiantes, profesores y grupos profesor-periodo sostienen la comparación.

La segunda tabla calcula la frontera de overlap: para cada posible punto de corte se pregunta cuántos contextos profesor-periodo contienen simultáneamente estudiantes de frecuencias adyacentes. Esta información fue la base para elegir originalmente \`1/2/3/4/5/6+\` como agrupación inferencial conservadora; el \`7+\` actual se conserva como resolución descriptiva y sensibilidad de alta frecuencia.


In [ ]:
support = pd.read_csv(TABLES_DIR / "01_exact_positive_visit_support.csv")
overlap_frontier = pd.read_csv(TABLES_DIR / "05_cut_overlap_frontier.csv")

display(Markdown("### Exact positive-frequency support"))
display(support.head(12))

display(Markdown("### Pair-overlap frontier used for the outcome-blind grouping decision"))
display(overlap_frontier)


The important feature is not only the number of students in a visit-frequency cell, but also whether adjacent frequency groups coexist across enough instructor-period contexts to support fixed-effect comparisons. Exact six is already relatively sparse; exact seven, eight, and nine are substantially smaller. Thus `7+` should be interpreted as a high-frequency descriptive tail, not as a uniquely optimal scientific threshold.


## 5. Descriptive attendance profile

The full manuscript reporting grid contains 5,393 students with zero visits and 1,234 students with at least one visit. Positive attendance is concentrated at low frequencies, with a smaller high-use tail.


### 5.1 Estadística descriptiva por frecuencia de asistencia

\`group_outcome_summary()\` calcula, para cada grupo de visitas, el tamaño muestral, la media y desviación estándar de \(Z\), intervalos de confianza de la media y la proporción de PASS con su intervalo correspondiente. En esta etapa **no se ajusta por covariables ni efectos fijos**; es la descripción marginal de los ocho grupos.

La finalidad es separar dos cosas que después pueden confundirse: el patrón visible en los datos y el patrón que permanece una vez que se compara a estudiantes dentro de contextos de calificación profesor-periodo y se ajusta por programa académico.


In [ ]:
descriptives = group_outcome_summary(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="Z_GRADE_PRIMARY",
    pass_col="PASS",
)

descriptives = descriptives.rename(columns={
    "outcome_mean": "mean_z",
    "outcome_sd": "sd_z",
    "outcome_ci95_low": "z_ci95_low",
    "outcome_ci95_high": "z_ci95_high",
    "binary_rate": "pass_rate",
    "binary_ci95_low": "pass_ci95_low",
    "binary_ci95_high": "pass_ci95_high",
})

display(
    descriptives[
        ["group", "n", "mean_z", "z_ci95_low", "z_ci95_high",
         "pass_rate", "pass_ci95_low", "pass_ci95_high"]
    ]
)


The descriptive profile already shows the main empirical pattern. Mean standardized performance is approximately \(-0.061\) at zero attendance and is positive in every attendance group. The corresponding observed pass rates are about 72.1% at zero visits, around 81% for one to three visits, and 89.1% for the 7+ group.

The sequence is **not monotone**. For example, the exact-six point estimate is below the five-visit estimate and below the 7+ estimate. This is one reason not to read visit count as a simple treatment dose.


### 5.2 Generación de las figuras descriptivas principales

Esta celda genera dos visualizaciones directamente desde las tablas agregadas del análisis. La primera muestra la distribución de **calificaciones numéricas observadas antes de imputar** y, en paralelo, las proporciones de resultados administrativos; la segunda utiliza el outcome continuo estandarizado y descompone su densidad por PASS, numeric fail, BV, RT y BA.

La comparación entre ambas figuras es metodológicamente importante porque permite ver qué parte de la forma de la distribución ya existe en las notas observadas y qué parte aparece al incorporar los resultados administrativos al outcome continuo.


In [ ]:
# Figure 1a: observed numeric grades and administrative outcomes before imputation.
render_paper_figure(paper21_figures.plot_observed_pre_imputation_structure)

# Figure 1: completed standardized outcome and outcome-state composition.
render_paper_figure(paper21_figures.plot_distribution_and_composition)


## 6. Primary benchmark: zero visits versus any attendance

The first estimand deliberately collapses all positive attendance into one group. This isolates **entry into the support system** from the later question of visit frequency among users.

The adjusted model includes instructor-period fixed effects and degree-programme indicators, with standard errors clustered by instructor-period. For the continuous outcome, the coefficient is in standard-deviation units. For PASS, the linear-probability specification yields an adjusted percentage-point difference.


### 6.1 Modelo base: cero visitas frente a cualquier asistencia

Esta función colapsa la exposición en dos grupos, \(0\) y \(1+\), y estima el benchmark principal mediante \`fixed_effect_group_comparisons()\`. Conceptualmente, para un outcome \(Y_i\), el modelo es

\[
Y_i=\alpha_{c(i)}+\delta_{d(i)}+\beta\,\mathbb{1}(K_i>0)+\varepsilon_i,
\]

donde \(\alpha_{c(i)}\) son efectos fijos de profesor × periodo y \(\delta_{d(i)}\) son indicadores del programa académico. Los errores estándar son cluster-robust al mismo nivel profesor-periodo.

Se ejecuta el mismo estimador para \(Z\) y para PASS. En el segundo caso se trata de un **linear probability model**, por lo que \(\beta\) se interpreta directamente como diferencia ajustada en probabilidad, expresada después en puntos porcentuales.


In [ ]:
def binary_attendance_benchmark(frame: pd.DataFrame, outcome_col: str):
    x = frame.copy()
    x["ATTENDANCE_BINARY"] = np.where(x["VISITS_CMAT_PERIOD"].gt(0), "1+", "0")
    pairwise, omnibus, info = fixed_effect_group_comparisons(
        x,
        group_col="ATTENDANCE_BINARY",
        group_order=["0", "1+"],
        outcome_col=outcome_col,
        fixed_effect_col="CLASSROOM_ID",
        cluster_col="CLASSROOM_ID",
        categorical_covariates=["CLAVECARRERA"],
        multiplicity_method="holm",
    )
    # The returned pair is 0 minus 1+, so reverse it for the manuscript convention.
    row = pairwise.iloc[0]
    return pd.Series({
        "outcome": outcome_col,
        "1+ minus 0": -float(row["estimate_group1_minus_group2"]),
        "ci95_low": -float(row["ci_high"]),
        "ci95_high": -float(row["ci_low"]),
        "p_value": float(row["p_raw"]),
        "N": int(row["n"]),
        "clusters": int(info.loc[0, "n_clusters"]),
    })

benchmark = pd.DataFrame([
    binary_attendance_benchmark(mu, "Z_GRADE_PRIMARY"),
    binary_attendance_benchmark(mu, "PASS"),
])
benchmark


The central benchmark is large: any recorded attendance is associated with approximately **+0.361 SD** in standardized final performance and **+15.3 percentage points** in pass probability. These are adjusted associations, not causal treatment effects.

Formally, the fitted comparison can be written schematically as

$$
Y_i=\alpha+\beta\,\mathbb{1}(\mathrm{CMAT}_i>0)
+\gamma_{\mathrm{instructor}\times\mathrm{period}}
+\delta_{\mathrm{degree}}+\varepsilon_i.
$$

The coefficient \(\beta\) describes an adjusted difference in observed outcomes. Because CMAT use is selected rather than randomized, it is not automatically equal to a causal potential-outcome contrast.


## 7. Frequency among users and the PPA threshold

The next question is whether outcomes form a staircase across 0, 1, 2, ..., 7+ visits. All 28 pairwise comparisons in the eight-group family are estimated with instructor-period fixed effects, degree-programme adjustment, cluster-robust uncertainty, and Holm correction for multiple testing.

The third visit receives special attention because it is the institutional PPA threshold, but an institutional threshold need not be an academic performance threshold.


### 7.1 Modelo de frecuencia completa y familia de contrastes pairwise

Aquí la exposición deja de ser binaria y se modelan simultáneamente los ocho grupos \`0/1/2/3/4/5/6/7+\`. Se ajustan dos modelos con la misma estructura de efectos fijos y covariables: uno para \(Z\) y otro para PASS.

A partir de un único modelo se construyen todas las diferencias por pares,

\[
\widehat{\Delta}_{kj}
=
\widehat{E}(Y\mid K=k,X)
-
\widehat{E}(Y\mid K=j,X),
\]

junto con intervalos cluster-robust. Como ocho grupos generan \( {8\choose2}=28 \) comparaciones, los valores \(p\) se corrigen con Holm para controlar el family-wise error rate. El omnibus prueba primero si existe alguna diferencia global entre frecuencias; los contrastes pairwise localizan después dónde se encuentra.


In [ ]:
z_pair, z_omnibus, z_info = fixed_effect_group_comparisons(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="Z_GRADE_PRIMARY",
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=["CLAVECARRERA"],
    multiplicity_method="holm",
)

pass_pair, pass_omnibus, pass_info = fixed_effect_group_comparisons(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="PASS",
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=["CLAVECARRERA"],
    multiplicity_method="holm",
)

display(Markdown("### Omnibus tests"))
display(pd.concat([
    z_omnibus.assign(outcome="standardized performance"),
    pass_omnibus.assign(outcome="PASS"),
], ignore_index=True))


### 7.2 Extracción de contrastes científicamente relevantes

El modelo anterior devuelve las 28 comparaciones posibles. Esta pequeña función no vuelve a estimar nada; únicamente localiza un par específico independientemente de cuál grupo aparezca como \`group1\` o \`group2\`.

Se utiliza para revisar de forma explícita tres contrastes vinculados con la hipótesis del paper: \(2\) vs \(3\) para evaluar si el umbral PPA coincide con un cambio académico, \(4\) vs \(5\) para inspeccionar el aumento descriptivo intermedio y \(6\) vs \(7+\) para comprobar si el perfil de alta frecuencia puede interpretarse como un salto adyacente.


In [ ]:
def get_pair(frame, a, b):
    hit = frame.loc[
        ((frame["group1"].astype(str) == str(a)) & (frame["group2"].astype(str) == str(b))) |
        ((frame["group1"].astype(str) == str(b)) & (frame["group2"].astype(str) == str(a)))
    ].copy()
    if hit.empty:
        raise KeyError(f"Pair {a} vs {b} not found")
    return hit

display(Markdown("### PPA threshold: 2 versus 3 visits"))
display(get_pair(z_pair, "2", "3"))

display(Markdown("### Descriptive later changes: 4 versus 5 and 6 versus 7+"))
display(pd.concat([
    get_pair(z_pair, "4", "5"),
    get_pair(z_pair, "6", "7+"),
], ignore_index=True))


The two-versus-three contrast is extremely small and statistically unresolved, so the third visit is not an observable performance discontinuity. The visually noticeable increases from four to five visits and from six to 7+ are also not statistically resolved after accounting for uncertainty and multiplicity.

This produces a central distinction:

$$
\text{institutional threshold at 3 visits}
\neq
\text{demonstrated academic threshold at 3 visits}.
$$

Likewise, visible changes later in the sequence should be described as **descriptive profile changes**, not causal jumps.


## 8. Why 7+ is nevertheless distinctive

The 7+ group has the highest descriptive mean standardized outcome and the highest observed pass rate. The binary PASS analysis provides stronger within-user evidence than the continuous outcome: the adjusted pass probability for 7+ exceeds the one-visit group by about 14.5 percentage points after Holm correction.

The manuscript also reports fixed-effect logistic **odds-ratio point estimates** for magnitude. Because the full logistic clustered covariance is rank-deficient in this design, multiplicity-aware PASS inference comes from the stable linear-probability model, while logistic odds ratios are treated as descriptive effect-size estimates.


### 8.1 PASS en alta frecuencia: LPM para inferencia y logit para magnitud

Esta celda separa deliberadamente **inferencia** y **escala de efecto**. El contraste \(1\) vs \(7+\) se toma primero del linear probability model, cuya covarianza cluster-robust es estable y permite aplicar Holm a la familia completa de comparaciones.

Después se ajusta un logit con los mismos efectos fijos y covariables,

\[
\operatorname{logit}P(\mathrm{PASS}_i=1)
=
\alpha_{c(i)}+\delta_{d(i)}+\beta_k,
\]

para convertir diferencias de frecuencia en **odds ratios**. En esta especificación la matriz de covarianza cluster-robust puede ser rank-deficient cuando el nivel de efecto fijo coincide con el clustering, por lo que los OR se utilizan como descriptores de magnitud y no como la fuente principal de significancia.


In [ ]:
display(Markdown("### 1 versus 7+ PASS contrast from the stable linear-probability model"))
display(get_pair(pass_pair, "1", "7+"))

logit_pair, logit_omnibus, logit_info = fixed_effect_logistic_group_comparisons(
    mu,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="PASS",
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=["CLAVECARRERA"],
    multiplicity_method="holm",
)

display(Markdown("### Logistic point estimate for 1 versus 7+"))
display(get_pair(logit_pair, "1", "7+"))


### 8.2 Dashboards de efectos y multiplicidad

Estas tres figuras convierten las salidas pairwise en matrices \(8\times8\). El primer dashboard representa magnitudes —diferencias ajustadas en \(Z\) y odds ratios de PASS—, mientras que los dos siguientes separan valores \(p\) crudos de valores \(p\) corregidos por Holm.

La línea discontinua se coloca entre 2 y 3 visitas para que el umbral institucional PPA pueda compararse visualmente con la estructura inferencial; el color representa magnitud o valor \(p\), pero la interpretación estadística proviene de los contrastes ajustados y no de la apariencia de la matriz.


In [ ]:
# Pairwise adjusted effect dashboard.
render_paper_figure(paper21_figures.plot_pairwise_effect_dashboard)

# Raw and Holm-adjusted pairwise p-value dashboards.
render_paper_figure(paper21_figures.plot_z_pvalue_dashboard)
render_paper_figure(paper21_figures.plot_pass_pvalue_dashboard)


The evidence therefore supports describing 7+ as a **distinctive high-frequency profile**, especially for PASS. It does not support the stronger claim that a causal threshold begins at the seventh visit, because the adjacent exact-six versus 7+ comparison is itself imprecise.


## 9. Adverse outcomes are not interchangeable

PASS/non-PASS hides substantively different endpoints. A non-passing attempt may end as:

- a completed numeric grade below 7.5,
- BV (voluntary withdrawal),
- RT (temporary withdrawal),
- BA (academic withdrawal).

The paper therefore distinguishes a **performance margin** from an **academic-management margin**. The second asks, conditional on not passing, how the adverse outcome is recorded.


### 9.1 Descomposición del resultado final en cinco estados

Hasta aquí PASS resumía todos los desenlaces adversos en una sola categoría. Esta celda abre el outcome en cinco estados mutuamente excluyentes:

\[
\{\mathrm{PASS},\;\mathrm{numeric}<7.5,\;\mathrm{BV},\;\mathrm{RT},\;\mathrm{BA}\}.
\]

El cálculo es puramente descriptivo: cuenta estudiantes por frecuencia y estado, divide por el tamaño total de cada grupo y reorganiza el resultado en una matriz. Esto permite distinguir si una tasa de no-aprobación similar proviene de más reprobaciones numéricas o de más terminaciones administrativas.


In [ ]:
composition = (
    mu.groupby(["P21_GROUP_7P", "ACADEMIC_OUTCOME_STATE_5"], observed=True)
      .size()
      .rename("n")
      .reset_index()
)
composition["share"] = composition.groupby("P21_GROUP_7P")["n"].transform(lambda s: s / s.sum())
composition_pivot = composition.pivot(
    index="P21_GROUP_7P", columns="ACADEMIC_OUTCOME_STATE_5", values="share"
).reindex(GROUP_ORDER).fillna(0)

composition_pivot


### 9.2 Academic-management margin condicionado a no aprobar

Aquí la muestra se restringe a estudiantes con \(\mathrm{PASS}=0\). Dentro de ese subconjunto se construyen contrastes binarios distintos: cualquier resultado administrativo vs reprobación numérica, BV vs reprobación numérica y RT vs reprobación numérica.

Cada contraste se estima nuevamente con efectos fijos profesor-periodo, programa académico y errores cluster-robust. La restricción condicional es importante: la pregunta ya no es “¿quién reprueba más?”, sino

\[
P(\text{tipo de desenlace adverso}\mid \mathrm{nonPASS},\;K).
\]

Las categorías que no pertenecen a un contraste particular se excluyen mediante valores faltantes, por ejemplo RT y BA al estimar BV vs numeric fail.


In [ ]:
nonpass = mu.loc[mu["PASS"].eq(0)].copy()

def management_benchmark(outcome_col: str):
    x = nonpass.loc[nonpass[outcome_col].notna()].copy()
    x["ATTENDANCE_BINARY"] = np.where(x["VISITS_CMAT_PERIOD"].gt(0), "1+", "0")
    pairwise, _, info = fixed_effect_group_comparisons(
        x,
        group_col="ATTENDANCE_BINARY",
        group_order=["0", "1+"],
        outcome_col=outcome_col,
        fixed_effect_col="CLASSROOM_ID",
        cluster_col="CLASSROOM_ID",
        categorical_covariates=["CLAVECARRERA"],
        multiplicity_method="holm",
    )
    row = pairwise.iloc[0]
    return pd.Series({
        "contrast": outcome_col,
        "1+ minus 0": -float(row["estimate_group1_minus_group2"]),
        "ci95_low": -float(row["ci_high"]),
        "ci95_high": -float(row["ci_low"]),
        "p_value": float(row["p_raw"]),
        "N": int(row["n"]),
    })

management = pd.DataFrame([
    management_benchmark("ADMINISTRATIVE_VS_NUMERIC_NONPASS"),
    management_benchmark("BV_VS_NUMERIC_NONPASS"),
    management_benchmark("RT_VS_NUMERIC_NONPASS"),
])
management


Among students who do not pass, CMAT users are more likely to end with an administrative outcome rather than a numeric failure. Crucially, that pattern is concentrated in **BV**. The corresponding RT-versus-numeric-failure contrast is essentially null.

This prevents the interpretation from collapsing all administrative outcomes into one behavioral mechanism. The records show a BV-specific composition difference; they do **not** show why students used BV, whether they knew particular institutional rules, or whether CMAT changed the withdrawal decision.


## 10. Numeric failure without administrative-outcome imputation

A useful robustness check removes BV, RT, and BA entirely and asks a simpler question among students with an observed numeric final grade:

$$
P(Y<7.5\mid \text{visit frequency}).
$$

If the broad zero-versus-positive difference survives here, it cannot be attributed solely to how administrative outcomes were numerically imputed into the continuous outcome.


### 10.1 Sensibilidad complete-case: reprobación numérica observada

Esta celda elimina completamente BV, RT y BA y conserva únicamente estudiantes con una calificación final numérica observada. Se define

\[
\mathrm{NUMERIC\_FAIL}=1-\mathrm{PASS},
\]

de manera que el outcome ya no depende de ningún valor imputado para resultados administrativos.

El resumen reporta \(n\), número de reprobaciones y probabilidad empírica de reprobación por grupo. Si la separación entre cero visitas y usuarios aparece también aquí, entonces esa señal no puede explicarse exclusivamente por la imputación utilizada en el outcome continuo principal.


In [ ]:
numeric = mu.loc[mu["GRADE_CLASS"].eq("numeric")].copy()
numeric["NUMERIC_FAIL"] = 1.0 - numeric["PASS"]

numeric_fail_summary = (
    numeric.groupby("P21_GROUP_7P", observed=True)["NUMERIC_FAIL"]
           .agg(n="size", failures="sum", probability="mean")
           .reindex(GROUP_ORDER)
)
numeric_fail_summary


### 10.2 Probabilidades ajustadas de reprobación mediante logit

Sobre la muestra numeric-only se estima un modelo logístico con efectos fijos profesor-periodo y programa académico. A partir de los coeficientes se calculan **probabilidades ajustadas** para cada frecuencia, en lugar de limitarse a presentar coeficientes en log-odds.

Conceptualmente,

\[
P(\mathrm{NUMERIC\_FAIL}_i=1\mid K_i,X_i)
=
\operatorname{logit}^{-1}
\left(
\alpha_{c(i)}+\delta_{d(i)}+\beta_{K_i}
\right).
\]

Esta transformación hace que la salida sea interpretable directamente como riesgo ajustado de terminar MU con una nota numérica menor a 7.5.


In [ ]:
adjusted_failure = fixed_effect_logistic_adjusted_probabilities(
    numeric,
    group_col="P21_GROUP_7P",
    group_order=GROUP_ORDER,
    outcome_col="NUMERIC_FAIL",
    fixed_effect_col="CLASSROOM_ID",
    cluster_col="CLASSROOM_ID",
    categorical_covariates=["CLAVECARRERA"],
)
adjusted_failure


The observed-numeric analysis preserves the same broad structure: completed numeric failure is much more common at zero attendance than among CMAT users. Among positive frequencies, however, there is no regular visit-by-visit failure gradient. This is another reason the paper separates the large **engagement margin** (0 vs 1+) from the weaker **frequency margin** among users.


## 11. Distributional question: one asymmetric distribution or two regimes?

A mean can hide substantial heterogeneity. The paper therefore asks whether each attendance group is better represented by a single asymmetric distribution or by a two-component Gaussian mixture.

For a two-Gaussian model,

$$
f(z)=\pi\,\phi(z;\mu_L,\sigma_L^2)
+(1-\pi)\,\phi(z;\mu_H,\sigma_H^2),
$$

where the labels \(L\) and \(H\) refer only to the lower-mean and higher-mean density components. They are **not** assumed to be literal latent student types.

The critical alternative is a one-component skew-normal distribution. Without this comparison, ordinary left-skewness could be mistaken for substantive bimodality.


### Heavy-model execution policy

The complete mixture analysis includes multistart Gaussian mixtures, a skew-normal comparison, parametric bootstrap, and repeated cross-validation. Those calculations are intentionally expensive. The repository already contains the verified aggregate outputs used by the manuscript.

Set `RUN_HEAVY_MODELS = True` below only when a full recomputation is desired. Otherwise, the notebook reads the canonical versioned outputs.


### 11.1 Reestimación opcional de los modelos de mezcla

Esta celda controla la parte computacionalmente más costosa del análisis. Cuando \`RUN_HEAVY_MODELS=True\`, se ejecuta \`run_paper21_mixture.py\`, que ajusta modelos Gaussian Mixture Model (GMM) de \(K=1,2,3\) componentes y las especificaciones skew-normal de comparación.

Para un GMM de dos componentes,

\[
f(z_i)
=
\pi_1\phi(z_i;\mu_1,\sigma_1^2)
+
\pi_2\phi(z_i;\mu_2,\sigma_2^2),
\qquad \pi_1+\pi_2=1.
\]

Los parámetros se estiman por máxima verosimilitud mediante **Expectation-Maximization (EM)**. En el E-step se calculan las responsabilidades posteriores

\[
r_{ik}
=
\frac{\pi_k\phi(z_i;\mu_k,\sigma_k^2)}
{\sum_j \pi_j\phi(z_i;\mu_j,\sigma_j^2)},
\]

y en el M-step esas responsabilidades se utilizan para actualizar pesos, medias y varianzas. El procedimiento se inicia desde múltiples puntos porque la likelihood de una mezcla puede tener máximos locales.

El runner complementa EM con BIC/ICL, bootstrap paramétrico para \(K=1\) vs \(K=2\), comparación contra skew-normal y validación predictiva repetida. Por defecto la notebook reutiliza los outputs agregados ya verificados para evitar recalcular cientos de ajustes en cada apertura.


In [ ]:
RUN_HEAVY_MODELS = False

if RUN_HEAVY_MODELS:
    subprocess.run(
        [
            sys.executable,
            str(REPO_ROOT / "code" / "run_paper21_mixture.py"),
            "--materias", str(REPO_ROOT / "data" / "controlled" / "Materias_pseudonymized.csv"),
            "--asesorias", str(REPO_ROOT / "data" / "controlled" / "Asesorias_pseudonymized.csv"),
            "--gmm-bootstrap", "199",
            "--shape-bootstrap", "199",
            "--shape-cv-folds", "5",
            "--shape-cv-repeats", "10",
        ],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print("Using versioned, controlled-run mixture outputs. Set RUN_HEAVY_MODELS=True to recompute them.")


### 11.2 Parámetros estimados de las mezclas gaussianas

Aquí se leen los parámetros producidos por EM para las dos especificaciones del outcome: **numeric complete-case** y **primary imputed**. Para cada grupo y componente se observan principalmente tres cantidades:

- \(\hat\pi_k\): peso estimado del componente;
- \(\hat\mu_k\): media del componente en unidades de \(Z\);
- \(\hat\sigma_k\): dispersión dentro del componente.

Los grupos 0–5 provienen del ajuste histórico principal y exact 6 / 7+ de la sensibilidad de cola; se combinan únicamente para reproducir la resolución actual del manuscrito. El orden “lower-performance / higher-performance” se asigna por la media estimada y no implica clases psicológicas o tipos fijos de estudiante.


In [ ]:
# Combine historical 0-5/6+ outputs with the exact-6 and 7+ sensitivity outputs
# exactly as the manuscript does when discussing the high-frequency tail.
cc_params = pd.read_csv(TABLES_DIR / "38_complete_case_gmm_two_component_parameters.csv")
imp_params = pd.read_csv(TABLES_DIR / "42_imputed_gmm_two_component_parameters.csv")
cc_tail = pd.read_csv(TABLES_DIR / "54_sensitivity_6_7plus_complete_case_gmm_two_component_parameters.csv")
imp_tail = pd.read_csv(TABLES_DIR / "58_sensitivity_6_7plus_imputed_gmm_two_component_parameters.csv")

display(Markdown("### Numeric complete-case two-Gaussian parameters"))
display(pd.concat([
    cc_params.loc[cc_params["group"].astype(str).isin(["0","1","2","3","4","5"])],
    cc_tail,
], ignore_index=True))

display(Markdown("### Imputed-outcome two-Gaussian parameters"))
display(pd.concat([
    imp_params.loc[imp_params["group"].astype(str).isin(["0","1","2","3","4","5"])],
    imp_tail,
], ignore_index=True))


### 11.3 ¿Bimodalidad real o simplemente asimetría?

Un GMM de dos componentes casi siempre puede ajustarse mejor in-sample que una distribución más simple porque tiene más parámetros. Por eso esta celda no interpreta automáticamente un mejor fit del GMM como evidencia de dos regímenes.

Se compara \(GMM_{K=2}\) con una **skew-normal de un solo componente**, que puede representar una distribución unimodal pero asimétrica. La decisión se apoya en tres diagnósticos complementarios: criterios de información como BIC, bootstrap paramétrico bajo la skew-normal como modelo nulo y log predictive density fuera de muestra mediante cross-validation repetida.

La comparación se realiza por separado para numeric complete-case y para el outcome imputado; si la evidencia de mezcla aparece sólo después de la imputación, eso indica que los estados administrativos están contribuyendo materialmente a la segunda forma distributiva.


In [ ]:
shape_cc = pd.read_csv(TABLES_DIR / "47_complete_case_skewnormal_vs_gmm.csv")
shape_imp = pd.read_csv(TABLES_DIR / "48_imputed_skewnormal_vs_gmm.csv")
shape_cc_tail = pd.read_csv(TABLES_DIR / "63_sensitivity_6_7plus_complete_case_skewnormal_vs_gmm.csv")
shape_imp_tail = pd.read_csv(TABLES_DIR / "64_sensitivity_6_7plus_imputed_skewnormal_vs_gmm.csv")

display(Markdown("### Shape-model comparison: observed numeric grades"))
display(shape_cc)
display(shape_cc_tail)

display(Markdown("### Shape-model comparison: imputed continuous outcome"))
display(shape_imp)
display(shape_imp_tail)


The observed numeric-grade result is deliberately conservative. A two-Gaussian model is clearly supported for some low-frequency groups, while several higher-frequency distributions are adequately represented by a single skew-normal. In particular, the apparent 7+ complete-case lower component is essentially an unstable extreme-tail fit rather than evidence for a stable second population.

The imputed outcome shows a clearer recurring lower component because administrative outcomes are numerically placed in the lower tail. This difference between observed and imputed specifications is itself scientifically informative: administrative course endings materially shape the distributional structure.


### 11.4 Visualización de los ajustes GMM y del peso del componente inferior

La primera figura superpone, para cada frecuencia, la KDE observada complete-case con las dos densidades gaussianas estimadas por EM y con su suma ponderada. Esto permite comprobar visualmente si el segundo componente corresponde a una estructura distribuida o únicamente a una observación extrema en la cola.

La segunda figura traza \(\hat\pi_L\), el peso del componente de menor media, tanto en complete-case como en el outcome imputado. El objetivo no es interpretar \(\pi_L\) como “proporción de malos estudiantes”, sino observar cómo cambia la masa asignada a la región de bajo desempeño cuando los desenlaces administrativos se incorporan al outcome.


In [ ]:
# Complete-case Gaussian-mixture ridgeline and lower-component weight comparison.
render_paper_figure(paper21_figures.plot_complete_case_gmm_ridgeline)
render_paper_figure(paper21_figures.plot_lower_component_weight)


## 12. What is inside the lower imputed component?

The mixture is not converted into hard student labels. Instead, posterior component responsibilities are aggregated softly by the observed final state. This preserves uncertainty about component membership.

Among CMAT users, the lower imputed component is strongly enriched in administrative outcomes, especially BV, whereas the higher component is overwhelmingly PASS. This links the distributional result to the separate academic-management result without claiming that the lower component is a psychological student type.


### 12.1 Composición de componentes usando responsabilidades posteriores

Un GMM no necesita asignar a cada estudiante de manera rígida a un componente. Si un estudiante tiene responsabilidades \((r_{iL},r_{iH})=(0.60,0.40)\), aporta 0.60 observaciones esperadas al componente inferior y 0.40 al superior.

Las tablas leídas aquí agregan esas responsabilidades por estado académico. Matemáticamente, para un estado \(s\),

\[
\widehat N_{ks}
=
\sum_i r_{ik}\,\mathbb{1}(S_i=s).
\]

Esta **soft classification** preserva la incertidumbre de pertenencia y permite preguntar qué proporción esperada de cada componente corresponde a PASS, numeric fail, BV, RT o BA sin crear etiquetas individuales deterministas.


In [ ]:
imputed_composition = pd.read_csv(TABLES_DIR / "44_imputed_gmm_soft_component_composition.csv")
imputed_tail_composition = pd.read_csv(TABLES_DIR / "60_sensitivity_6_7plus_imputed_gmm_soft_component_composition.csv")

display(imputed_composition)
display(imputed_tail_composition)


Aggregated across positive-attendance groups in the manuscript's validated analysis, the lower imputed component is approximately 55.5% administrative outcomes, including about 49.5% BV, whereas the higher component is approximately 97% PASS.

The appropriate interpretation is therefore:

> the completed outcome contains a lower distributional component strongly enriched in adverse administrative endings, especially BV.

The inappropriate interpretation would be that the model has discovered fixed categories such as “good students” and “bad students.”


## 13. Instructor-level clustering sensitivity

The primary specification clusters by instructor-period because that is also the fixed-effect grading context. As a dependence sensitivity, the paper re-clusters by instructor while retaining instructor-period fixed effects.

The broad zero-versus-positive benchmark remains nearly unchanged. Positive-frequency omnibus tests become somewhat more suggestive, but the result still does not become a clean adjacent visit-by-visit staircase.


### 13.1 Sensibilidad al nivel de clustering

El modelo principal usa efectos fijos profesor-periodo y también agrupa la covarianza por profesor-periodo. Como un mismo profesor puede aparecer en varios periodos, esta sensibilidad mantiene los mismos efectos fijos pero permite correlación de errores a través de todos los cursos impartidos por un profesor.

No cambia el estimando ni los coeficientes de interés; cambia la matriz de varianza-covarianza utilizada para construir errores estándar, intervalos y pruebas. Si las conclusiones generales sobreviven a este clustering más agregado, dependen menos de asumir independencia entre periodos del mismo profesor.


In [ ]:
instructor_benchmark = pd.read_csv(TABLES_DIR / "29_instructor_cluster_benchmark_0_vs_1plus.csv")
instructor_omnibus = pd.read_csv(TABLES_DIR / "28_instructor_cluster_omnibus.csv")

display(instructor_benchmark)
display(instructor_omnibus)


## 14. Discussion

The results support a layered interpretation of CMAT attendance.

First, the strongest and most stable empirical separation is between **zero attendance and any recorded attendance**. Every positive-frequency group lies above the zero group on adjusted standardized performance, and the broad benchmark is also large for PASS.

Second, frequency among users is not a smooth dose-response curve. The PPA-compatible first three visits are similar to one another, and the third visit does not coincide with an academic discontinuity. Later descriptive changes, including four-to-five and six-to-7+, are visible but imprecise.

Third, the 7+ group is a meaningful high-frequency profile because it has the highest observed mean and pass rate and because its adjusted PASS probability exceeds the one-visit group after multiplicity correction. This is evidence of a distinctive observed profile, not proof that the seventh visit itself causes the difference.

Fourth, adverse outcomes differ in **type**, not only in frequency. CMAT users who do not pass are relatively less likely to finish with a numeric failing grade and relatively more likely to have BV, while RT does not show the same adjusted contrast. This pattern is compatible with broader academic engagement or institutional navigation, but those mechanisms are not directly measured.

Finally, the distributional analysis shows why observed numeric grades and the imputed completed outcome must both be reported. Administrative outcomes materially sharpen the lower component of the imputed distribution. The stronger bimodality in the completed outcome therefore cannot be interpreted as a universal two-type structure among students.


## 15. Limits and identification

The principal limitation is self-selection. Students are not randomly assigned to CMAT, and visit frequency can be related to prior preparation, academic difficulty, motivation, schedule flexibility, persistence, advice seeking, and other unmeasured characteristics.

Timing is also endogenous. Students may attend because difficulties emerge, return because the service is useful, stop attending because they no longer need help, or have fewer opportunities to accumulate visits because they withdraw earlier.

PPA further complicates the behavioral meaning of the first three visits because the records do not identify which visits were specifically motivated by participation credit. Thus the third visit is not a valid regression-discontinuity cutoff or instrument in the present data.

The high-frequency cells are small, especially exact six and the exact counts above six before pooling, so uncertainty is visibly wider in the upper tail. Mixture components are density approximations rather than identified latent classes, and the numerical values assigned to BV/RT/BA are unobserved imputations whose role must remain transparent.

Consequently, the target estimand of the present notebook is descriptive/associational:

$$
E[Y\mid K=k, X] - E[Y\mid K=j, X],
$$

after the documented adjustment, rather than the causal potential-outcome contrast

$$
E[Y(k)-Y(j)].
$$


## 16. Conclusion

The full attendance profile contains substantially more structure than a simple user/non-user indicator, but it does not support a monotone dose-response interpretation.

The most robust result is the large **0 versus 1+** separation. Within positive attendance, the first one to three visits are difficult to interpret as purely voluntary because of PPA, and there is no observable break exactly at the third visit. The 7+ group is the clearest high-frequency profile, especially for PASS, while adjacent high-frequency contrasts remain imprecise.

The distributional extension adds an important second result: once administrative outcomes are incorporated into the completed continuous outcome, a lower performance component becomes much clearer and is strongly enriched in BV and other adverse endings. Observed numeric grades alone provide a more heterogeneous and less universally bimodal picture.

The defensible synthesis is therefore:

$$
\boxed{
\text{CMAT frequency is not a simple treatment dose;}
\quad
0\text{ vs }1+\text{ dominates,}
\quad
7+\text{ is distinctive,}
\quad
\text{and adverse-outcome composition matters.}
}
$$


## 17. Full canonical regeneration

All seven manuscript figures above are already **constructed and rendered inline** by the notebook. The cell below is separate: it is only for regenerating the repository's canonical aggregate CSV tables and saved vector-PDF artifacts from the controlled pseudonymized inputs.

It is intentionally not executed automatically because the full mixture bootstrap and repeated cross-validation are computationally expensive.


### 17.1 Regeneración completa de tablas y figuras

Esta es la ruta de reproducción integral del paper. Al activar \`RUN_FULL_PIPELINE=True\`, \`paper_build.py --tables --figures\` ejecuta secuencialmente la auditoría de soporte, los modelos de outcomes, los modelos de mezcla y finalmente todas las recetas gráficas.

A diferencia de las celdas anteriores, que permiten estudiar cada paso de forma interactiva, esta celda verifica el pipeline como una unidad: datos controlados \(\rightarrow\) tablas agregadas \(\rightarrow\) figuras de publicación. La opción permanece apagada por defecto debido al costo del bootstrap y de la cross-validation de mezclas.


In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    subprocess.run(
        [sys.executable, str(PAPER_DIR / "paper_build.py"), "--tables", "--figures"],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print(
        "Full pipeline not run. Set RUN_FULL_PIPELINE=True to regenerate all canonical "
        "aggregate tables and publication figures from the controlled pseudonymized inputs."
    )


## 18. Provenance and source files

The notebook is a presentation layer over the same analysis stack as the manuscript. For audit or modification, the canonical sources are:

- `paper/sections/01.tex`–`04.tex`: manuscript text;
- `paper/docs/analysis/ANALYSIS_PLAN.md`: analysis contract;
- `paper/docs/analysis/VISIT_GROUPING_DECISION.md`: outcome-blind grouping history;
- `paper/docs/results/PRELIMINARY_RESULTS.md`: verified result summary;
- `paper/docs/results/MIXTURE_ANALYSIS_RESULTS.md`: mixture-model validation and interpretation;
- `paper/docs/interpretation/ACADEMIC_MANAGEMENT_HYPOTHESIS.md`: mechanism guardrails;
- `code/run_paper21_outcomes.py`: main outcome analysis;
- `code/run_paper21_mixture.py`: numeric-failure and mixture analysis;
- `code/figures_paper21.py`: publication figures;
- `paper/references.bib`: bibliography.

The notebook should be updated when the canonical manuscript or analysis changes; scientific methods should continue to live in `cmat_analysis`, not be forked into notebook-only helper implementations.


### 18.1 Bibliografía exacta utilizada por el manuscrito

La última celda no realiza inferencia estadística. Lee directamente \`paper/references.bib\` para que la notebook conserve trazabilidad con las referencias del manuscrito y permita inspeccionar qué fuente corresponde a cada argumento de la revisión de literatura.

Se muestra el archivo BibTeX sin transformarlo, de modo que la notebook no mantenga una segunda bibliografía que pueda divergir del paper.


In [ ]:
# Optional: show the exact bibliography used by the manuscript.
references = (PAPER_DIR / "references.bib").read_text(encoding="utf-8")
print(references)
